# Download Cost Data to Lakehouse (Incremental)

This notebook uses the **Generate Cost Details Report** API (async) to download
cost data as a CSV file. It uses **incremental refresh** — only the last N days
(configured via `COST_LOOKBACK_DAYS` in `.env`) are fetched and merged into the
existing tables.

**How it works:**
1. POST to generate a cost details report for the last N days
2. Poll until the report is ready
3. Download the CSV from the blob URL
4. Parse into Spark DataFrames
5. **Dimension tables**: merge new rows with existing (no duplicates lost)
6. **Fact table (cost_record)**: delete rows for fetched dates, then append fresh data

**Prerequisites:**
- A Microsoft Fabric Lakehouse attached to this notebook
- A Service Principal with **Cost Management Reader** on your subscription
- A `.env` file uploaded to Lakehouse Files (run `GetKeys.ps1` to create it)


In [ ]:
# ─── Load credentials ─────────────────────────────────────────────────────────
# Priority: Azure Key Vault (pipeline-safe) > .env file (local dev)
import os

def load_env(filepath):
    """Parse a .env file and set os.environ variables."""
    if not os.path.exists(filepath):
        return False
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()
    return True

# Try .env file (local dev / manual runs)
env_loaded = False
for p in [
    os.path.join(os.path.dirname(os.getcwd()), ".env"),
    os.path.join(os.getcwd(), ".env"),
    "/lakehouse/default/Files/.env",
]:
    if load_env(p):
        print(f"Loaded .env from: {p}")
        env_loaded = True
        break

# Resolve credentials: Key Vault first, then .env fallback
KEY_VAULT_URL = os.environ.get("KEY_VAULT_URL", "")

if KEY_VAULT_URL:
    from notebookutils import mssparkutils
    AZURE_TENANT_ID       = mssparkutils.credentials.getSecret(KEY_VAULT_URL, "AZURE-TENANT-ID")
    AZURE_CLIENT_ID       = mssparkutils.credentials.getSecret(KEY_VAULT_URL, "AZURE-CLIENT-ID")
    AZURE_CLIENT_SECRET   = mssparkutils.credentials.getSecret(KEY_VAULT_URL, "AZURE-CLIENT-SECRET")
    AZURE_SUBSCRIPTION_ID = mssparkutils.credentials.getSecret(KEY_VAULT_URL, "AZURE-SUBSCRIPTION-ID")
    print(f"Credentials loaded from Key Vault: {KEY_VAULT_URL}")
elif env_loaded:
    AZURE_TENANT_ID       = os.environ["AZURE_TENANT_ID"]
    AZURE_CLIENT_ID       = os.environ["AZURE_CLIENT_ID"]
    AZURE_CLIENT_SECRET   = os.environ["AZURE_CLIENT_SECRET"]
    AZURE_SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
    print("WARNING: Using .env file credentials. For pipelines, set KEY_VAULT_URL.")
else:
    raise RuntimeError(
        "No credentials found. Either:\n"
        "  1. Set KEY_VAULT_URL env var pointing to your Azure Key Vault, or\n"
        "  2. Upload .env file to Lakehouse Files (run GetKeys.ps1 first)"
    )

# Lookback days — how many days of cost data to fetch/refresh
COST_LOOKBACK_DAYS = int(os.environ.get("COST_LOOKBACK_DAYS", "7"))

print(f"Subscription: {AZURE_SUBSCRIPTION_ID}")
print(f"Tenant: {AZURE_TENANT_ID}")
print(f"Client: {AZURE_CLIENT_ID[:8]}...")
print(f"Lookback: {COST_LOOKBACK_DAYS} days")

# Table names in Lakehouse
TABLE_COST_RECORD    = "cost_record"
TABLE_SUBSCRIPTION   = "subscription"
TABLE_RESOURCE_GROUP = "resource_group"
TABLE_RESOURCE       = "resource"
TABLE_METER          = "meter_category"
TABLE_SERVICE        = "service"


In [ ]:
import requests
import json
import time
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 1. Authenticate with Azure (OAuth2 Client Credentials)


In [ ]:
token_url = f"https://login.microsoftonline.com/{AZURE_TENANT_ID}/oauth2/v2.0/token"

token_body = {
    "grant_type":    "client_credentials",
    "client_id":     AZURE_CLIENT_ID,
    "client_secret": AZURE_CLIENT_SECRET,
    "scope":         "https://management.azure.com/.default",
}

token_response = requests.post(token_url, data=token_body)
token_response.raise_for_status()
access_token = token_response.json()["access_token"]

print("Authentication successful.")


## 2. Fetch Cost Data (one day at a time)

The Cost Details Report API has a **1-month max** date range limit. To support
any lookback window (7, 30, 90+ days), we request **one report per day** and
combine the results. Each day's CSV is saved to Lakehouse Files for reference.


In [ ]:
# ─── Fetch cost data in ≤30-day chunks (single-threaded, with 429 retry) ──────
import os as _os

scope = f"/subscriptions/{AZURE_SUBSCRIPTION_ID}"
report_url = (
    f"https://management.azure.com{scope}"
    f"/providers/Microsoft.CostManagement/generateCostDetailsReport"
    f"?api-version=2024-08-01"
)

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type":  "application/json",
}

today = datetime.utcnow().date()
csv_dir = "/lakehouse/default/Files/cost_daily"
_os.makedirs(csv_dir, exist_ok=True)

CHUNK_MAX_DAYS = 30  # API limit: max 1 month per request
MAX_429_RETRIES = 5  # how many times to retry on throttling

def fetch_chunk(start_date, end_date):
    """Request, poll, and download a cost report for a date range. Returns CSV bytes or None."""
    start_str = start_date.strftime("%Y-%m-%d")
    end_str = end_date.strftime("%Y-%m-%d")
    report_body = {
        "metric": "ActualCost",
        "timePeriod": {"start": start_str, "end": end_str},
    }

    # POST with 429 retry
    for retry in range(MAX_429_RETRIES + 1):
        resp = requests.post(report_url, headers=headers, json=report_body)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", "60"))
            print(f"    429 throttled — waiting {wait}s (retry {retry+1}/{MAX_429_RETRIES})")
            time.sleep(wait)
            continue
        break
    else:
        print(f"    ERROR: still throttled after {MAX_429_RETRIES} retries")
        return None

    if resp.status_code == 204:
        return None
    if resp.status_code not in (200, 202):
        print(f"    ERROR {resp.status_code}: {resp.text[:200]}")
        return None

    # Poll if async
    download_url = None
    if resp.status_code == 202:
        poll_url = resp.headers.get("Location")
        retry_after = int(resp.headers.get("Retry-After", 20))
        for attempt in range(90):
            time.sleep(retry_after)
            poll_resp = requests.get(poll_url, headers=headers)
            if poll_resp.status_code == 200:
                result = poll_resp.json()
                status = result.get("status", "").lower()
                if status in ("completed", "succeeded"):
                    blobs = result.get("manifest", {}).get("blobs", [])
                    if blobs:
                        download_url = blobs[0].get("blobLink")
                    break
                elif status == "failed":
                    print(f"    FAILED: {json.dumps(result, indent=2)[:300]}")
                    return None
                else:
                    print(f"    polling {attempt+1}/90 (status={status}, retry={retry_after}s)")
            elif poll_resp.status_code == 202:
                retry_after = int(poll_resp.headers.get("Retry-After", 20))
                print(f"    polling {attempt+1}/90 (HTTP 202, retry={retry_after}s)")
            elif poll_resp.status_code == 429:
                wait = int(poll_resp.headers.get("Retry-After", "60"))
                print(f"    polling {attempt+1}/90 — 429 throttled, waiting {wait}s")
                time.sleep(wait)
            else:
                print(f"    Poll error: HTTP {poll_resp.status_code}")
                return None
        else:
            print(f"    TIMEOUT after 90 poll attempts")
            return None
    elif resp.status_code == 200:
        result = resp.json()
        blobs = result.get("manifest", {}).get("blobs", [])
        if blobs:
            download_url = blobs[0].get("blobLink")

    if not download_url:
        return None

    csv_resp = requests.get(download_url)
    csv_resp.raise_for_status()
    return csv_resp.content

# ─── Build chunks of ≤30 days ────────────────────────────────────────────────
chunks = []
start = today - timedelta(days=COST_LOOKBACK_DAYS)
while start < today:
    end = min(start + timedelta(days=CHUNK_MAX_DAYS - 1), today - timedelta(days=1))
    chunks.append((start, end))
    start = end + timedelta(days=1)

print(f"Fetching {COST_LOOKBACK_DAYS} days in {len(chunks)} chunk(s) of ≤{CHUNK_MAX_DAYS} days each\n")

# ─── Sequential fetch per chunk ──────────────────────────────────────────────
all_csv_paths = []
total_bytes = 0
errors = []

for idx, (chunk_start, chunk_end) in enumerate(chunks, 1):
    label = f"{chunk_start} to {chunk_end}"
    days_in_chunk = (chunk_end - chunk_start).days + 1
    print(f"  Chunk {idx}/{len(chunks)}: {label} ({days_in_chunk} days) ... ", end="", flush=True)

    csv_data = fetch_chunk(chunk_start, chunk_end)
    csv_path = f"{csv_dir}/cost_{chunk_start}_{chunk_end}.csv"

    if csv_data and len(csv_data) > 50:
        with open(csv_path, "wb") as f:
            f.write(csv_data)
        all_csv_paths.append(csv_path)
        total_bytes += len(csv_data)
        print(f"✓ {len(csv_data):,} bytes")
    else:
        if _os.path.exists(csv_path):
            _os.remove(csv_path)
        errors.append(label)
        print("✗ no data")

print(f"\nDone: {len(all_csv_paths)}/{len(chunks)} chunks downloaded, {total_bytes:,} bytes total")
if errors:
    print(f"\n⚠ {len(errors)} chunk(s) with no data:")
    for e in errors:
        print(f"  {e}")

## 3. Load All CSVs into Spark


In [ ]:
# ─── Load all daily CSVs into a single DataFrame ──────────────────────────────
if not all_csv_paths:
    raise RuntimeError("No cost data downloaded. Check API errors above.")

# Read all CSVs at once (Spark handles schema unification)
cost_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/cost_daily/cost_*.csv")
)

row_count = cost_df.count()
col_count = len(cost_df.columns)
print(f"Loaded: {row_count:,} rows x {col_count} columns from {len(all_csv_paths)} files")
print(f"\nColumns:\n  {cost_df.columns}")


## 4. Preview Data


In [ ]:
# Preview
display(cost_df.limit(20))


## 5. Detect Columns


## 6. Write Dimension and Fact Tables to Lakehouse (Incremental)

**Dimension tables** (subscription, resource_group, resource, meter_category, service):
merge new rows with existing — new keys are inserted, existing keys are left as-is.

**Fact table** (cost_record): delete all rows for the fetched date range,
then append the fresh data. This ensures no duplicates while preserving
historical data outside the lookback window.


In [ ]:
# ─── Detect available columns (case-insensitive) ─────────────────────────────
# The Cost Details Report CSV uses camelCase (e.g. "subscriptionName", "meterCategory")
# which differs from the Query API's PascalCase. We build a case-insensitive lookup.
available = cost_df.columns
col_map = {c.lower(): c for c in available}  # lowercase -> actual name

def pick(candidates):
    """Return the first column name found (case-insensitive), or None."""
    for c in candidates:
        actual = col_map.get(c.lower())
        if actual:
            return actual
    return None

COL_SUB_ID       = pick(["SubscriptionId", "subscriptionId"])
COL_SUB_NAME     = pick(["SubscriptionName", "subscriptionName"])
COL_RG           = pick(["ResourceGroup", "resourceGroup", "ResourceGroupName", "resourceGroupName"])
COL_RES_ID       = pick(["ResourceId", "resourceId", "InstanceId", "instanceId"])
COL_RES_TYPE     = pick(["ResourceType", "resourceType"])
COL_RES_LOC      = pick(["ResourceLocation", "resourceLocation"])
COL_METER_CAT    = pick(["MeterCategory", "meterCategory"])
COL_METER_SUB    = pick(["MeterSubCategory", "meterSubCategory"])
COL_METER_NAME   = pick(["MeterName", "meterName"])
COL_UOM          = pick(["UnitOfMeasure", "unitOfMeasure"])
COL_SVC          = pick(["ConsumedService", "consumedService"])
COL_CHARGE       = pick(["ChargeType", "chargeType"])
COL_COST         = pick(["CostInBillingCurrency", "costInBillingCurrency", "PreTaxCost", "preTaxCost", "Cost", "cost"])
COL_QTY          = pick(["Quantity", "quantity", "UsageQuantity", "usageQuantity"])
COL_DATE         = pick(["date", "Date", "UsageDateTime", "usageDateTime",
                         "servicePeriodStartDate", "ServicePeriodStartDate",
                         "servicePeriodEndDate", "ServicePeriodEndDate",
                         "BillingPeriodStartDate", "billingPeriodStartDate",
                         "UsageDate", "usageDate"])
COL_CURRENCY     = pick(["BillingCurrency", "billingCurrency", "Currency", "currency"])

# If we found a date column, check its actual data to determine parsing
if COL_DATE:
    sample_date = cost_df.select(F.col(COL_DATE)).filter(F.col(COL_DATE).isNotNull()).limit(1).collect()
    if sample_date:
        print(f"\nDate column '{COL_DATE}' sample value: '{sample_date[0][0]}' (type: {type(sample_date[0][0]).__name__})")

print("\nColumn mapping:")
for label, col in [("SubscriptionId", COL_SUB_ID), ("SubscriptionName", COL_SUB_NAME),
                   ("ResourceGroup", COL_RG), ("ResourceId", COL_RES_ID),
                   ("ResourceType", COL_RES_TYPE), ("ResourceLocation", COL_RES_LOC),
                   ("MeterCategory", COL_METER_CAT), ("MeterSubCategory", COL_METER_SUB),
                   ("MeterName", COL_METER_NAME), ("UnitOfMeasure", COL_UOM),
                   ("ConsumedService", COL_SVC), ("ChargeType", COL_CHARGE),
                   ("Cost", COL_COST), ("Quantity", COL_QTY),
                   ("Date", COL_DATE), ("Currency", COL_CURRENCY)]:
    status = "found" if col else "MISSING"
    print(f"  {label}: {col or 'N/A'} ({status})")


In [ ]:
# ─── Helper: merge dimension table (insert new keys, keep existing) ────────────
from delta.tables import DeltaTable

def merge_dimension(new_df, table_name, key_col):
    """Merge new rows into an existing Delta dimension table.
    New keys are inserted; existing keys are left unchanged."""
    try:
        existing = DeltaTable.forName(spark, table_name)
        existing.alias("old").merge(
            new_df.alias("new"),
            f"old.{key_col} = new.{key_col}"
        ).whenNotMatchedInsertAll().execute()
        print(f"  {table_name}: merged ({spark.table(table_name).count()} total rows)")
    except Exception:
        # Table doesn't exist yet — create it
        new_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table_name)
        print(f"  {table_name}: created ({new_df.count()} rows)")

# ─── Subscription dimension ───────────────────────────────────────────────────
if COL_SUB_ID and COL_SUB_NAME:
    sub_df = cost_df.select(
        F.col(COL_SUB_ID).alias("subscriptionId"),
        F.col(COL_SUB_NAME).alias("subscriptionName"),
    ).distinct()
else:
    sub_df = spark.createDataFrame(
        [(AZURE_SUBSCRIPTION_ID, "")], ["subscriptionId", "subscriptionName"])
merge_dimension(sub_df, TABLE_SUBSCRIPTION, "subscriptionId")

# ─── Resource Group dimension ─────────────────────────────────────────────────
if COL_RG and COL_SUB_ID:
    rg_df = cost_df.select(
        F.concat(F.col(COL_SUB_ID), F.lit("/"), F.col(COL_RG)).alias("resourceGroupId"),
        F.col(COL_RG).alias("resourceGroupName"),
        F.col(COL_SUB_ID).alias("subscriptionId"),
    ).distinct().filter(F.col(COL_RG) != "").filter(F.col(COL_RG).isNotNull())
    merge_dimension(rg_df, TABLE_RESOURCE_GROUP, "resourceGroupId")

# ─── Resource dimension ──────────────────────────────────────────────────────
if COL_RES_ID:
    res_cols = [F.col(COL_RES_ID).alias("resourceId")]
    if COL_RES_TYPE:
        res_cols.append(F.col(COL_RES_TYPE).alias("resourceType"))
    else:
        res_cols.append(
            F.regexp_extract(
                F.col(COL_RES_ID),
                r"providers/([^/]+/[^/]+)",
                1
            ).alias("resourceType")
        )
    if COL_RG: res_cols.append(F.col(COL_RG).alias("resourceGroupName"))
    if COL_RG and COL_SUB_ID:
        res_cols.append(F.concat(F.col(COL_SUB_ID), F.lit("/"), F.col(COL_RG)).alias("resourceGroupId"))
    if COL_RES_LOC: res_cols.append(F.col(COL_RES_LOC).alias("location"))
    res_df = cost_df.select(*res_cols).distinct().filter(F.col("resourceId") != "").filter(F.col("resourceId").isNotNull())
    merge_dimension(res_df, TABLE_RESOURCE, "resourceId")

# ─── Meter Category dimension ────────────────────────────────────────────────
if COL_METER_CAT:
    meter_cols = [F.col(COL_METER_CAT).alias("meterCategory")]
    if COL_METER_SUB: meter_cols.append(F.col(COL_METER_SUB).alias("meterSubCategory"))
    else: meter_cols.append(F.lit("").alias("meterSubCategory"))
    if COL_METER_NAME: meter_cols.append(F.col(COL_METER_NAME).alias("meterName"))
    else: meter_cols.append(F.lit("").alias("meterName"))
    if COL_UOM: meter_cols.append(F.col(COL_UOM).alias("unitOfMeasure"))
    else: meter_cols.append(F.lit("").alias("unitOfMeasure"))

    meter_df = cost_df.select(*meter_cols).distinct().filter(F.col("meterCategory") != "").filter(F.col("meterCategory").isNotNull())
    meter_df = meter_df.withColumn("meterId", F.md5(F.concat_ws("|",
        F.col("meterCategory"), F.col("meterSubCategory"), F.col("meterName"))))
    merge_dimension(meter_df, TABLE_METER, "meterId")

# ─── Service dimension ───────────────────────────────────────────────────────
if COL_SVC:
    svc_df = cost_df.select(
        F.col(COL_SVC).alias("serviceName"),
        F.col(COL_CHARGE).alias("chargeType") if COL_CHARGE else F.lit("Usage").alias("chargeType"),
    ).distinct().filter(F.col("serviceName") != "").filter(F.col("serviceName").isNotNull())
    svc_df = svc_df.withColumn("serviceId", F.md5(F.concat_ws("|",
        F.col("serviceName"), F.col("chargeType"))))
    merge_dimension(svc_df, TABLE_SERVICE, "serviceId")


In [ ]:
# ─── Cost Record fact table (incremental: delete fetched dates, then append) ──
fact_base = cost_df
has_meter = False
has_svc = False

# Check if meter_category table exists before joining
try:
    meter_lookup = spark.table(TABLE_METER).select(
        F.col("meterId").alias("_meterId"),
        F.col("meterCategory").alias("_meterCategory"),
        F.col("meterSubCategory").alias("_meterSubCategory"),
        F.col("meterName").alias("_meterName"),
    )
    if COL_METER_CAT and COL_METER_SUB and COL_METER_NAME:
        fact_base = fact_base.join(
            meter_lookup,
            (fact_base[COL_METER_CAT] == meter_lookup["_meterCategory"]) &
            (fact_base[COL_METER_SUB] == meter_lookup["_meterSubCategory"]) &
            (fact_base[COL_METER_NAME] == meter_lookup["_meterName"]),
            "left"
        )
        has_meter = True
except Exception:
    print(f"  Skipping meter join — {TABLE_METER} table not available")

# Check if service table exists before joining
try:
    svc_lookup = spark.table(TABLE_SERVICE).select(
        F.col("serviceId").alias("_serviceId"),
        F.col("serviceName").alias("_serviceName"),
        F.col("chargeType").alias("_chargeType"),
    )
    if COL_SVC and COL_CHARGE:
        fact_base = fact_base.join(
            svc_lookup,
            (fact_base[COL_SVC] == svc_lookup["_serviceName"]) &
            (fact_base[COL_CHARGE] == svc_lookup["_chargeType"]),
            "left"
        )
        has_svc = True
except Exception:
    print(f"  Skipping service join — {TABLE_SERVICE} table not available")

# Select final columns (no costRecordId — we'll regenerate after merge)
fact_cols = []
if COL_SUB_ID: fact_cols.append(F.col(COL_SUB_ID).alias("subscriptionId"))
if COL_RG and COL_SUB_ID:
    fact_cols.append(F.concat(F.col(COL_SUB_ID), F.lit("/"), F.col(COL_RG)).alias("resourceGroupId"))
if COL_RES_ID: fact_cols.append(F.col(COL_RES_ID).alias("resourceId"))
if has_meter: fact_cols.append(F.col("_meterId").alias("meterId"))
else: fact_cols.append(F.lit(None).alias("meterId"))
if has_svc: fact_cols.append(F.col("_serviceId").alias("serviceId"))
else: fact_cols.append(F.lit(None).alias("serviceId"))
if COL_DATE:
    date_expr = F.coalesce(
        F.to_date(F.col(COL_DATE), "MM/dd/yyyy"),
        F.to_date(F.col(COL_DATE), "yyyy-MM-dd"),
        F.to_date(F.col(COL_DATE), "yyyy-MM-dd'T'HH:mm:ss'Z'"),
        F.to_date(F.col(COL_DATE), "dd/MM/yyyy"),
        F.to_date(F.col(COL_DATE)),
        F.to_date(F.substring(F.col(COL_DATE), 1, 10)),
    )
    fact_cols.append(date_expr.alias("usageDate"))
if COL_COST: fact_cols.append(F.col(COL_COST).cast("double").alias("preTaxCost"))
if COL_QTY: fact_cols.append(F.col(COL_QTY).cast("double").alias("quantity"))
if COL_UOM: fact_cols.append(F.col(COL_UOM).alias("unitOfMeasure"))
if COL_CHARGE: fact_cols.append(F.col(COL_CHARGE).alias("chargeType"))
if COL_RES_LOC: fact_cols.append(F.col(COL_RES_LOC).alias("location"))
if COL_CURRENCY: fact_cols.append(F.col(COL_CURRENCY).alias("Currency"))

new_facts = fact_base.select(*fact_cols)

# Determine the date range we fetched
fetched_dates = new_facts.select(F.min("usageDate").alias("min_dt"), F.max("usageDate").alias("max_dt")).collect()[0]
fetch_min = str(fetched_dates["min_dt"])
fetch_max = str(fetched_dates["max_dt"])
print(f"  Fetched date range: {fetch_min} to {fetch_max} ({new_facts.count():,} rows)")

# Delete existing rows for fetched dates, then append
try:
    existing = DeltaTable.forName(spark, TABLE_COST_RECORD)
    # Delete rows in the fetched date range
    existing.delete(f"usageDate >= '{fetch_min}' AND usageDate <= '{fetch_max}'")
    print(f"  Deleted existing rows for {fetch_min} to {fetch_max}")
    # Append new data (with fresh costRecordId)
    new_facts_with_id = new_facts.withColumn("costRecordId", F.monotonically_increasing_id())
    new_facts_with_id.write.mode("append").format("delta").saveAsTable(TABLE_COST_RECORD)
    total = spark.table(TABLE_COST_RECORD).count()
    print(f"  {TABLE_COST_RECORD}: appended {new_facts.count():,} rows ({total:,} total)")
except Exception:
    # Table doesn't exist yet — create it
    new_facts_with_id = new_facts.withColumn("costRecordId", F.monotonically_increasing_id())
    new_facts_with_id.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(TABLE_COST_RECORD)
    print(f"  {TABLE_COST_RECORD}: created ({new_facts_with_id.count():,} rows)")

# Print summary of all tables
print("\nTables summary:")
for t in [TABLE_SUBSCRIPTION, TABLE_RESOURCE_GROUP, TABLE_RESOURCE, TABLE_METER, TABLE_SERVICE, TABLE_COST_RECORD]:
    try:
        count = spark.table(t).count()
        print(f"  {t}: {count:,} rows")
    except Exception:
        print(f"  {t}: not created (source columns missing from CSV)")


## 7. Verify — Sample Data


In [ ]:
# Summary — per resource per day cost breakdown
cost_table = spark.table(TABLE_COST_RECORD)
print(f"Cost record table: {cost_table.count():,} rows, {len(cost_table.columns)} columns")
print(f"Columns: {cost_table.columns}\n")

# Show cost per resource per day
if "resourceId" in cost_table.columns and "usageDate" in cost_table.columns:
    summary = (
        cost_table
        .filter(F.col("resourceId").isNotNull() & (F.col("resourceId") != ""))
        .groupBy("resourceId", "usageDate")
        .agg(
            F.sum("preTaxCost").alias("dailyCost"),
            F.sum("quantity").alias("totalQuantity"),
        )
        .orderBy(F.desc("dailyCost"))
    )
    print(f"Unique resource-day combinations: {summary.count():,}")
    display(summary.limit(30))
else:
    display(cost_table.limit(20))


In [ ]:
print(f"\nNotebook completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tables are ready in your Lakehouse for Power BI and Ontology creation.")
